# Bear Data Cleaning Pipeline

This notebook loads and cleans bear individual data from the Excel file 'Dades os_1996_2024.xlsx'.
It extracts the first sheet containing individual bear information and exports it to CSV format.

## 1. Configuration and Imports

In [92]:
!pip install openpyxl

In [93]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

import warnings
warnings.filterwarnings('ignore')

# Configuration: Paths and file names
NOTEBOOK_DIR = Path.cwd()
INPUT_FILE = NOTEBOOK_DIR / "Dades os_1996_2024.xlsx"
OUTPUT_DIR = NOTEBOOK_DIR / "cleaned_data"

# Create output directory if it doesn't exist
OUTPUT_DIR.mkdir(exist_ok=True)

# Excel sheet names
INDIVIDUALS_SHEET = "Individus_Totals_2024"  # Adjust if needed

print(f"Input file: {INPUT_FILE}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Input file exists: {INPUT_FILE.exists()}")

Input file: /home/aniol-garriga-torra-boss/Escriptori/ANIOL/UNI/4t Carrera/TFG/TFG-pirineus_raster/notebooks/Dades os_1996_2024.xlsx
Output directory: /home/aniol-garriga-torra-boss/Escriptori/ANIOL/UNI/4t Carrera/TFG/TFG-pirineus_raster/notebooks/cleaned_data
Input file exists: True


## 2. Load and Inspect Data

In [94]:
# Load all sheet names to identify the individuals sheet
xls = pd.ExcelFile(INPUT_FILE)
print("Available sheet names:")
print(xls.sheet_names)

# Load the first sheet (individuals data)
df_raw = pd.read_excel(INPUT_FILE, sheet_name=0)

print(f"\nShape of raw data: {df_raw.shape}")
print(f"\nColumn names:")
print(df_raw.columns.tolist())
print(f"\nFirst few rows:")
df_raw.head()

Available sheet names:
['Individus_Totals_2024', 'Dades GPS', 'Total_os_bru_1996_2024']

Shape of raw data: (194, 28)

Column names:
['Code commun / Código común', 'Nom Ours / Nombre Oso', 'Génotype Antagene / Genotipo Antagene', 'Genotype UAB', 'Sexe / Sexo', 'Année Naissance / Año de nacimiento', 'Mère / Madre', 'Codi mare', 'Père / Padre', 'Codi pare', 'Année de Mortalité / Ano de Mortalidad', 'Année de disparition supposée / Año de la presunta desaparición', 'Age / Edad', 'Detectado en 2022', 'Detectado en 2023', 'Detectado en 2024', datetime.datetime(2024, 2, 1, 0, 0), datetime.datetime(2024, 3, 1, 0, 0), datetime.datetime(2024, 4, 1, 0, 0), datetime.datetime(2024, 5, 1, 0, 0), datetime.datetime(2024, 6, 1, 0, 0), datetime.datetime(2024, 7, 1, 0, 0), datetime.datetime(2024, 8, 1, 0, 0), datetime.datetime(2024, 9, 1, 0, 0), datetime.datetime(2024, 10, 1, 0, 0), datetime.datetime(2024, 11, 1, 0, 0), datetime.datetime(2024, 12, 1, 0, 0), 'Total 2024']

First few rows:


,Code commun / Código común,Nom Ours / Nombre Oso,Génotype Antagene / Genotipo Antagene,Genotype UAB,Sexe / Sexo,Année Naissance / Año de nacimiento,Mère / Madre,Codi mare,Père / Padre,Codi pare,...,2024-04-01 00:00:00,2024-05-01 00:00:00,2024-06-01 00:00:00,2024-07-01 00:00:00,2024-08-01 00:00:00,2024-09-01 00:00:00,2024-10-01 00:00:00,2024-11-01 00:00:00,2024-12-01 00:00:00,Total 2024
0,NaN,Papillon,NaN,NaN,M,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
1,NaN,Cannelle,S2-PYR6,NaN,F,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
2,NaN,Camille / Aspe-Ouest,S1-PYR4,Camille,M,1998,Cannelle,Sense codi,Papillon,Sense codi,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
3,NaN,Ourson mort,NaN,NaN,M,2000,Cannelle,Sense codi,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
4,F001,Ziva,S8-SLO13,NaN,F,1990,Slovène,Sense codi,Slovène,Sense codi,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0


In [95]:
# Diagnostic: Check column types
print("Column types and values:")
for i, col in enumerate(df_raw.columns):
    print(f"{i:2d}. Type: {type(col).__name__:15s} | Value: {str(col)[:60]}")

Column types and values:
 0. Type: str             | Value: Code commun / Código común
 1. Type: str             | Value: Nom Ours / Nombre Oso
 2. Type: str             | Value: Génotype Antagene / Genotipo Antagene
 3. Type: str             | Value: Genotype UAB
 4. Type: str             | Value: Sexe / Sexo
 5. Type: str             | Value: Année Naissance / Año de nacimiento
 6. Type: str             | Value: Mère / Madre
 7. Type: str             | Value: Codi mare
 8. Type: str             | Value: Père / Padre
 9. Type: str             | Value: Codi pare
10. Type: str             | Value: Année de Mortalité / Ano de Mortalidad
11. Type: str             | Value: Année de disparition supposée / Año de la presunta desaparic
12. Type: str             | Value: Age / Edad
13. Type: str             | Value: Detectado en 2022
14. Type: str             | Value: Detectado en 2023
15. Type: str             | Value: Detectado en 2024
16. Type: datetime        | Value: 2024-02-01 00:00:00
1

In [96]:
# Check data types and missing values
print("Data types:")
print(df_raw.dtypes)
print(f"\nMissing values:")
print(df_raw.isnull().sum())
print(f"\nBasic statistics:")
df_raw.describe()

Data types:
Code commun / Código común                                             str
Nom Ours / Nombre Oso                                                  str
Génotype Antagene / Genotipo Antagene                                  str
Genotype UAB                                                           str
Sexe / Sexo                                                            str
Année Naissance / Año de nacimiento                                 object
Mère / Madre                                                           str
Codi mare                                                              str
Père / Padre                                                           str
Codi pare                                                              str
Année de Mortalité / Ano de Mortalidad                             float64
Année de disparition supposée / Año de la presunta desaparición    float64
Age / Edad                                                         float64
Detectado en 

,Année de Mortalité / Ano de Mortalidad,Année de disparition supposée / Año de la presunta desaparición,Age / Edad,Detectado en 2022,Detectado en 2023,Detectado en 2024,2024-02-01 00:00:00,2024-03-01 00:00:00,2024-04-01 00:00:00,2024-05-01 00:00:00,2024-06-01 00:00:00,2024-07-01 00:00:00,2024-08-01 00:00:00,2024-09-01 00:00:00,2024-10-01 00:00:00,2024-11-01 00:00:00,2024-12-01 00:00:00,Total 2024
count,35.000000,36.000000,121.000000,143.000000,160.000000,183.000000,10.000000,15.000000,24.000000,38.000000,42.000000,58.000000,55.000000,41.000000,38.000000,11.000000,3.000000,182.000000
mean,2013.885714,2016.777778,5.057851,0.531469,0.518750,1.049180,1.100000,1.600000,2.250000,3.131579,2.738095,2.448276,2.254545,2.097561,3.131579,1.727273,1.666667,4.494505
std,7.745099,5.986228,5.359256,0.500763,0.501217,7.075168,0.316228,1.352247,1.481773,3.094643,2.479767,2.027637,1.797304,1.578051,3.362520,1.009050,0.577350,7.187887
min,1997.000000,2002.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000
25%,2008.000000,2013.750000,1.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.500000,0.000000
50%,2017.000000,2018.500000,3.000000,1.000000,1.000000,1.000000,1.000000,1.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,1.500000,1.000000,2.000000,1.000000
75%,2020.000000,2022.000000,8.000000,1.000000,1.000000,1.000000,1.000000,1.500000,3.000000,4.000000,3.750000,3.000000,3.000000,3.000000,4.500000,3.000000,2.000000,6.000000
max,2022.000000,2023.000000,27.000000,1.000000,1.000000,96.000000,2.000000,6.000000,6.000000,16.000000,11.000000,10.000000,11.000000,8.000000,16.000000,3.000000,2.000000,42.000000


## 3. Data Cleaning and Standardization

In [97]:
column_mapping = {
    'Code commun / Código común': 'code',
    'Nom Ours / Nombre Oso': 'name',
    'Génotype Antagene / Genotipo Antagene': 'genotype',
    'Genotype UAB': 'genotype_uab',
    'Sexe / Sexo': 'sex',
    'Année Naissance / Año de nacimiento': 'born_year',
    'Mère / Madre': 'mum_name',
    'Codi mare': 'mum_code',
    'Père / Padre': 'father_name',
    'Codi pare': 'father_code',
    'Année de Mortalité / Ano de Mortalidad': 'mortality_year',
    'Année de disparition supposée / Año de la presunta desaparición': 'suposed_desaparition_year',
    'Age / Edad': 'age',
    'Detectado en 2022': 'detected_2022',
    'Detectado en 2023': 'detected_2023',
    'Detectado en 2024': 'detected_2024',
}

# Map datetime columns to month names
datetime_to_month = {
    2: 'num_detections_feb_24',
    3: 'num_detections_mar_24',
    4: 'num_detections_apr_24',
    5: 'num_detections_may_24',
    6: 'num_detections_jun_24',
    7: 'num_detections_jul_24',
    8: 'num_detections_aug_24',
    9: 'num_detections_sep_24',
    10: 'num_detections_oct_24',
    11: 'num_detections_nov_24',
    12: 'num_detections_dec_24',
}

# Find and map datetime columns (they're datetime.datetime objects)
for col in df_raw.columns:
    if isinstance(col, datetime):
        month = col.month
        if month in datetime_to_month:
            column_mapping[col] = datetime_to_month[month]

# Find and map the total column
for col in df_raw.columns:
    if isinstance(col, str) and 'Total' in col:
        column_mapping[col] = 'num_detections_total_24'

print("Column mapping created:")
for original, mapped in column_mapping.items():
    if isinstance(original, datetime):
        col_str = f"Timestamp({original.month:02d}/2024)"
    else:
        col_str = str(original)[:50]
    print(f"  {col_str:50s} → {mapped}")

# Get only columns that exist in the dataframe, preserving their order
columns_to_select = [col for col in df_raw.columns if col in column_mapping.keys()]
print(f"\nColumns found: {len(columns_to_select)}/{len(column_mapping)}")

Column mapping created:
  Code commun / Código común                         → code
  Nom Ours / Nombre Oso                              → name
  Génotype Antagene / Genotipo Antagene              → genotype
  Genotype UAB                                       → genotype_uab
  Sexe / Sexo                                        → sex
  Année Naissance / Año de nacimiento                → born_year
  Mère / Madre                                       → mum_name
  Codi mare                                          → mum_code
  Père / Padre                                       → father_name
  Codi pare                                          → father_code
  Année de Mortalité / Ano de Mortalidad             → mortality_year
  Année de disparition supposée / Año de la presunta → suposed_desaparition_year
  Age / Edad                                         → age
  Detectado en 2022                                  → detected_2022
  Detectado en 2023                                  → dete

In [98]:
df_clean = df_raw[columns_to_select].copy()

# Rename columns according to mapping
df_clean = df_clean.rename(columns=column_mapping)

# Filter to keep only valid individual records (rows 2-183 in Excel = indices 1-182 in Python)
print(f"Data shape before filtering: {df_clean.shape}")
df_clean = df_clean.iloc[:182]  # Keep only rows 2-183 from Excel (1-based indexing)
print(f"Data shape after filtering to valid individuals: {df_clean.shape}")
print(f"  Kept rows: 1-183 (Excel indexing) = indices 0-182 (Python indexing)")

# Reorder columns to the exact order specified
desired_column_order = [
    'code', 'name', 'genotype', 'genotype_uab', 'sex', 'born_year',
    'mum_name', 'mum_code', 'father_name', 'father_code',
    'mortality_year', 'suposed_desaparition_year', 'age',
    'detected_2022', 'detected_2023', 'detected_2024',
    'num_detections_feb_24', 'num_detections_mar_24', 'num_detections_apr_24',
    'num_detections_may_24', 'num_detections_jun_24', 'num_detections_jul_24',
    'num_detections_aug_24', 'num_detections_sep_24', 'num_detections_oct_24',
    'num_detections_nov_24', 'num_detections_dec_24', 'num_detections_total_24'
]

# Select and reorder
available_columns = [col for col in desired_column_order if col in df_clean.columns]
df_clean = df_clean[available_columns]

print(f"\nDataframe shape: {df_clean.shape}")
print(f"\nFinal column order ({len(df_clean.columns)} columns):")
for i, col in enumerate(df_clean.columns, 1):
    print(f"  {i:2d}. {col}")

Data shape before filtering: (194, 28)
Data shape after filtering to valid individuals: (182, 28)
  Kept rows: 1-183 (Excel indexing) = indices 0-182 (Python indexing)

Dataframe shape: (182, 28)

Final column order (28 columns):
   1. code
   2. name
   3. genotype
   4. genotype_uab
   5. sex
   6. born_year
   7. mum_name
   8. mum_code
   9. father_name
  10. father_code
  11. mortality_year
  12. suposed_desaparition_year
  13. age
  14. detected_2022
  15. detected_2023
  16. detected_2024
  17. num_detections_feb_24
  18. num_detections_mar_24
  19. num_detections_apr_24
  20. num_detections_may_24
  21. num_detections_jun_24
  22. num_detections_jul_24
  23. num_detections_aug_24
  24. num_detections_sep_24
  25. num_detections_oct_24
  26. num_detections_nov_24
  27. num_detections_dec_24
  28. num_detections_total_24


In [99]:
# Fill missing codes for the first 4 rows with correct bear names
# Using .loc[] for reliable pandas assignment
df_clean.loc[0, 'code'] = 'Papillon'
df_clean.loc[1, 'code'] = 'Cannelle'
df_clean.loc[2, 'code'] = 'Camille'
df_clean.loc[3, 'code'] = 'Ourson'


print("✓ Codes assigned to first 4 rows:")
print(df_clean.loc[1:4, 'code'].to_string())

✓ Codes assigned to first 4 rows:
1    Cannelle
2     Camille
3      Ourson
4        F001


In [100]:
df_clean.head()

,code,name,genotype,genotype_uab,sex,born_year,mum_name,mum_code,father_name,father_code,...,num_detections_apr_24,num_detections_may_24,num_detections_jun_24,num_detections_jul_24,num_detections_aug_24,num_detections_sep_24,num_detections_oct_24,num_detections_nov_24,num_detections_dec_24,num_detections_total_24
0,Papillon,Papillon,NaN,NaN,M,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
1,Cannelle,Cannelle,S2-PYR6,NaN,F,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
2,Camille,Camille / Aspe-Ouest,S1-PYR4,Camille,M,1998,Cannelle,Sense codi,Papillon,Sense codi,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
3,Ourson,Ourson mort,NaN,NaN,M,2000,Cannelle,Sense codi,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
4,F001,Ziva,S8-SLO13,NaN,F,1990,Slovène,Sense codi,Slovène,Sense codi,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0


In [101]:
# Data type conversions and cleaning

# String columns - handle null/NaN values
string_columns = ['code', 'name', 'genotype', 'genotype_uab', 'mum_name', 'mum_code', 'father_name', 'father_code']
for col in string_columns:
    if col in df_clean.columns:
        # Convert to string and handle NaN
        df_clean[col] = df_clean[col].astype(str).replace('nan', pd.NA)
        # Strip whitespace
        df_clean[col] = df_clean[col].apply(lambda x: x.strip() if pd.notna(x) else pd.NA)

# Sex should be categorical (M, F, I) - uppercase
if 'sex' in df_clean.columns:
    df_clean['sex'] = df_clean['sex'].astype(str).str.upper().replace('nan', pd.NA).str.strip()
    # Validate sex values
    valid_sex = df_clean['sex'].isin(['M', 'F', 'I']) | df_clean['sex'].isna()
    if not valid_sex.all():
        invalid = df_clean[~valid_sex]['sex'].unique()
        print(f"Warning: Invalid sex values found: {invalid}")

# Year columns (as string to preserve format)
year_columns = ['born_year', 'mortality_year', 'suposed_desaparition_year']
for col in year_columns:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype(str).replace('nan', pd.NA).str.strip()

# Age column (integer or null)
if 'age' in df_clean.columns:
    df_clean['age'] = pd.to_numeric(df_clean['age'], errors='coerce').astype('Int64')

# Binary detection columns (0 or 1)
detection_columns = ['detected_2022', 'detected_2023', 'detected_2024']
for col in detection_columns:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce').astype('Int64')
        # Validate binary values
        valid_binary = df_clean[col].isin([0, 1]) | df_clean[col].isna()
        if not valid_binary.all():
            print(f"Warning: {col} contains non-binary values")

# Monthly detection count columns (integer or null)
monthly_cols = [col for col in df_clean.columns if col.startswith('num_detections_') and col != 'num_detections_total_24']
for col in monthly_cols:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce').astype('Int64')

# Total detections column (integer)
if 'num_detections_total_24' in df_clean.columns:
    df_clean['num_detections_total_24'] = pd.to_numeric(df_clean['num_detections_total_24'], errors='coerce').astype('Int64')

print("✓ Data type conversions completed")
print(f"\nData types:")
print(df_clean.dtypes)

['F?']
Length: 1, dtype: str
✓ Data type conversions completed

Data types:
code                           str
name                           str
genotype                       str
genotype_uab                   str
sex                            str
born_year                      str
mum_name                       str
mum_code                       str
father_name                    str
father_code                    str
mortality_year                 str
suposed_desaparition_year      str
age                          Int64
detected_2022                Int64
detected_2023                Int64
detected_2024                Int64
num_detections_feb_24        Int64
num_detections_mar_24        Int64
num_detections_apr_24        Int64
num_detections_may_24        Int64
num_detections_jun_24        Int64
num_detections_jul_24        Int64
num_detections_aug_24        Int64
num_detections_sep_24        Int64
num_detections_oct_24        Int64
num_detections_nov_24        Int64
num_detections

In [102]:
# Clean data: Remove question marks and spaces from STRING columns only
print("=== CLEANING DATA: Removing question marks and spaces ===\n")

# For string columns only, remove '?' and spaces
for col in df_clean.columns:
    if df_clean[col].dtype == 'str':  # String columns only
        # Remove question marks and spaces
        df_clean[col] = df_clean[col].apply(
            lambda x: str(x).replace('?', '').replace(' ', '') if pd.notna(x) else x
        )

print("✓ Cleaned: Removed question marks (?) and spaces from STRING columns only")
print(df_clean.head())

=== CLEANING DATA: Removing question marks and spaces ===

✓ Cleaned: Removed question marks (?) and spaces from STRING columns only
       code                name  genotype genotype_uab sex born_year  \
0  Papillon            Papillon       NaN          NaN   M       NaN   
1  Cannelle            Cannelle   S2-PYR6          NaN   F       NaN   
2   Camille  Camille/Aspe-Ouest   S1-PYR4      Camille   M      1998   
3    Ourson          Oursonmort       NaN          NaN   M      2000   
4      F001                Ziva  S8-SLO13          NaN   F      1990   

   mum_name   mum_code father_name father_code  ... num_detections_apr_24  \
0       NaN        NaN         NaN         NaN  ...                  <NA>   
1       NaN        NaN         NaN         NaN  ...                  <NA>   
2  Cannelle  Sensecodi    Papillon   Sensecodi  ...                  <NA>   
3  Cannelle  Sensecodi         NaN         NaN  ...                  <NA>   
4   Slovène  Sensecodi     Slovène   Sensecodi  .

In [103]:
# Data Validation and Quality Checks
print("=== DATA VALIDATION REPORT ===\n")

# Check for required code column
if 'code' in df_clean.columns:
    missing_codes = df_clean['code'].isna().sum()
    print(f"Missing codes: {missing_codes}")
    if missing_codes > 0:
        print(f"  Rows with missing codes: {df_clean[df_clean['code'].isna()].index.tolist()}")

# Check for duplicates
duplicate_codes = df_clean['code'].duplicated().sum()
print(f"Duplicate codes: {duplicate_codes}")
if duplicate_codes > 0:
    print(f"  Duplicate codes: {df_clean[df_clean['code'].duplicated(keep=False)]['code'].unique().tolist()}")

# Check sex distribution
if 'sex' in df_clean.columns:
    print(f"\nSex distribution:")
    print(df_clean['sex'].value_counts(dropna=False))

# Check detection columns
detection_cols = ['detected_2022', 'detected_2023', 'detected_2024']
detection_cols_present = [col for col in detection_cols if col in df_clean.columns]
if detection_cols_present:
    print(f"\nDetection summary:")
    for col in detection_cols_present:
        present = (df_clean[col] == 1).sum()
        print(f"  {col}: {present} individuals detected")

# Check monthly detection counts sum to total
if 'num_detections_total_24' in df_clean.columns:
    monthly_cols = [col for col in df_clean.columns if col.startswith('num_detections_') and col != 'num_detections_total_24']
    monthly_sum = df_clean[monthly_cols].sum(axis=1)
    total_col = df_clean['num_detections_total_24']
    
    # Compare (handle NaN cases)
    comparison = (monthly_sum == total_col) | (monthly_sum.isna() & total_col.isna())
    mismatches = (~comparison).sum()
    print(f"\nMonthly detections sum validation:")
    print(f"  Rows where monthly sum matches total: {comparison.sum()}")
    print(f"  Rows with mismatches: {mismatches}")
    if mismatches > 0:
        print(f"  Using calculated sum as total (Excel formulas may have been source)")

# Missing value summary
print(f"\nMissing values per column:")
missing_summary = df_clean.isnull().sum()
#missing_summary = missing_summary[missing_summary > 0]
if len(missing_summary) > 0:
    print(missing_summary)
else:
    print("  No missing values")

print(f"\nTotal rows: {len(df_clean)}")
print(f"Total columns: {len(df_clean.columns)}")

=== DATA VALIDATION REPORT ===

Missing codes: 0
Duplicate codes: 0

Sex distribution:
sex
F    85
M    73
I    24
Name: count, dtype: int64

Detection summary:
  detected_2022: 76 individuals detected
  detected_2023: 83 individuals detected
  detected_2024: 96 individuals detected

Monthly detections sum validation:
  Rows where monthly sum matches total: 182
  Rows with mismatches: 0

Missing values per column:
code                           0
name                         115
genotype                      31
genotype_uab                 145
sex                            0
born_year                      2
mum_name                       2
mum_code                       2
father_name                   25
father_code                   25
mortality_year               147
suposed_desaparition_year    146
age                           61
detected_2022                 39
detected_2023                 22
detected_2024                  0
num_detections_feb_24        172
num_detections_mar_24

In [104]:
# Correct name typos: Mellba -> Melba
mellba_rows = df_clean[df_clean['name'] == 'Mellba']
if len(mellba_rows) > 0:
    df_clean.loc[df_clean['name'] == 'Mellba', 'name'] = 'Melba'
    print(f"✓ Corrected name typos:")
    print(f"  'Mellba' → 'Melba' ({len(mellba_rows)} row(s))")
else:
    print("No 'Mellba' entries found in the dataset")


✓ Corrected name typos:
  'Mellba' → 'Melba' (1 row(s))


## 4. Export Cleaned Data

In [105]:
# Create the 'disappeared' column based on mortality_year and suposed_desaparition_year
# disappeared = 1 if either mortality_year or suposed_desaparition_year is not null, else 0

df_clean['disappeared'] = (
    (df_clean['mortality_year'].notna()) | 
    (df_clean['suposed_desaparition_year'].notna())
).astype(int)

print("✓ Column 'disappeared' created")
print(f"\nDisappeared distribution:")
print(df_clean['disappeared'].value_counts().sort_index())
print(f"\nColumn 'disappeared' added with {df_clean['disappeared'].sum()} disappeared individuals out of {len(df_clean)}")


✓ Column 'disappeared' created

Disappeared distribution:
disappeared
0    111
1     71
Name: count, dtype: int64

Column 'disappeared' added with 71 disappeared individuals out of 182


In [106]:
# Export individual bear data to CSV
output_file = OUTPUT_DIR / "bear_individuals.csv"

df_clean.to_csv(output_file, index=False)

print(f"✓ Cleaned data exported to: {output_file}")
print(f"  - Rows: {len(df_clean)}")
print(f"  - Columns: {len(df_clean.columns)}")
print(f"\nColumn list:")
for i, col in enumerate(df_clean.columns, 1):
    print(f"  {i:2d}. {col}")

✓ Cleaned data exported to: /home/aniol-garriga-torra-boss/Escriptori/ANIOL/UNI/4t Carrera/TFG/TFG-pirineus_raster/notebooks/cleaned_data/bear_individuals.csv
  - Rows: 182
  - Columns: 29

Column list:
   1. code
   2. name
   3. genotype
   4. genotype_uab
   5. sex
   6. born_year
   7. mum_name
   8. mum_code
   9. father_name
  10. father_code
  11. mortality_year
  12. suposed_desaparition_year
  13. age
  14. detected_2022
  15. detected_2023
  16. detected_2024
  17. num_detections_feb_24
  18. num_detections_mar_24
  19. num_detections_apr_24
  20. num_detections_may_24
  21. num_detections_jun_24
  22. num_detections_jul_24
  23. num_detections_aug_24
  24. num_detections_sep_24
  25. num_detections_oct_24
  26. num_detections_nov_24
  27. num_detections_dec_24
  28. num_detections_total_24
  29. disappeared


## 5. Preview of Cleaned Data

## 6. Extract GPS Data from Second Sheet

In [107]:
# Load the second sheet (GPS female bears with cubs)
df_gps_raw = pd.read_excel(INPUT_FILE, sheet_name=1)

print("GPS Data - Second Sheet")
print(f"Shape: {df_gps_raw.shape}")
print(f"\nColumn names:")
for i, col in enumerate(df_gps_raw.columns, 1):
    print(f"  {i:2d}. {col}")
print(f"\nFirst few rows:")
df_gps_raw.head()


GPS Data - Second Sheet
Shape: (4629, 28)

Column names:
   1. ID_obs
   2. Bear_name
   3. Sex
   4. Age
   5. Age_class
   6. Year
   7. Date_GMT
   8. Time_GMT
   9. Tracking_system
  10. System
  11. Datum
  12. Zone
  13. Coordinate_system
  14. X
  15. Y
  16. x_long
  17. y_lat
  18. Country
  19. Region
  20. Activity
  21. Altitude
  22. Temp
  23. With_cubs_estimated
  24. N_cubs_estimated
  25. ID_cub1
  26. ID_cub2
  27. ID_cub3
  28. Remarks

First few rows:


,ID_obs,Bear_name,Sex,Age,Age_class,Year,Date_GMT,Time_GMT,Tracking_system,System,...,Region,Activity,Altitude,Temp,With_cubs_estimated,N_cubs_estimated,ID_cub1,ID_cub2,ID_cub3,Remarks
0,36119,Hvala,F,8,Adult,2007,2007-05-01,03:01:00,GPS,UTM,...,Haute-Garonne,NaN,NaN,NaN,<6month,2,Pollen,Bambou,-,NaN
1,36120,Hvala,F,8,Adult,2007,2007-05-01,15:02:00,GPS,UTM,...,Haute-Garonne,NaN,NaN,NaN,<6month,2,Pollen,Bambou,-,NaN
2,36121,Hvala,F,8,Adult,2007,2007-05-01,18:03:00,GPS,UTM,...,Haute-Garonne,NaN,NaN,NaN,<6month,2,Pollen,Bambou,-,NaN
3,36122,Hvala,F,8,Adult,2007,2007-05-01,21:03:00,GPS,UTM,...,Haute-Garonne,NaN,NaN,NaN,<6month,2,Pollen,Bambou,-,NaN
4,36123,Hvala,F,8,Adult,2007,2007-05-02,09:02:00,GPS,UTM,...,Haute-Garonne,NaN,NaN,NaN,<6month,2,Pollen,Bambou,-,NaN


In [108]:
# Process GPS data: convert column names to lowercase
df_gps = df_gps_raw.copy()

# Convert all column names to lowercase
df_gps.columns = df_gps.columns.str.lower()

print("✓ Column names converted to lowercase")
print(f"\nProcessed column names:")
for i, col in enumerate(df_gps.columns, 1):
    print(f"  {i:2d}. {col}")


✓ Column names converted to lowercase

Processed column names:
   1. id_obs
   2. bear_name
   3. sex
   4. age
   5. age_class
   6. year
   7. date_gmt
   8. time_gmt
   9. tracking_system
  10. system
  11. datum
  12. zone
  13. coordinate_system
  14. x
  15. y
  16. x_long
  17. y_lat
  18. country
  19. region
  20. activity
  21. altitude
  22. temp
  23. with_cubs_estimated
  24. n_cubs_estimated
  25. id_cub1
  26. id_cub2
  27. id_cub3
  28. remarks


In [109]:
# Join with bear individuals data to get bear_code
# Create lookup dataframe from df_clean (name -> code mapping)
bear_lookup = df_clean[['code', 'name']].drop_duplicates()
print(f"Bear lookup table: {len(bear_lookup)} unique bear-code pairs")
print(bear_lookup.head())

# Merge GPS data with bear codes
# Match df_gps.bear_name with df_clean.name -> get df_clean.code
df_gps = df_gps.merge(
    bear_lookup,
    left_on='bear_name',
    right_on='name',
    how='left'
)

# Rename the merged 'code' column to 'bear_code'
df_gps = df_gps.rename(columns={'code': 'bear_code'})

# Remove the 'name' column (duplicate of bear_name)
df_gps = df_gps.drop(columns=['name'])

print(f"\n✓ Join completed")
print(f"  Total GPS records: {len(df_gps)}")
print(f"  GPS records with bear_code: {df_gps['bear_code'].notna().sum()}")
print(f"  GPS records without bear_code (unmatched): {df_gps['bear_code'].isna().sum()}")


Bear lookup table: 182 unique bear-code pairs
       code                name
0  Papillon            Papillon
1  Cannelle            Cannelle
2   Camille  Camille/Aspe-Ouest
3    Ourson          Oursonmort
4      F001                Ziva

✓ Join completed
  Total GPS records: 4629
  GPS records with bear_code: 4629
  GPS records without bear_code (unmatched): 0


In [115]:
# Reorder columns: bear_code should be after id_obs and before bear_name
current_cols = df_gps.columns.tolist()

# Remove bear_code from current position
new_col_order = [col for col in current_cols if col != 'bear_code']

# Insert bear_code after id_obs (position 1)
insert_position = new_col_order.index('bear_name')
new_col_order.insert(insert_position, 'bear_code')

# Reorder dataframe
df_gps = df_gps[new_col_order]

print("✓ Columns reordered")
print(f"\nFinal column order ({len(df_gps.columns)} columns):")
for i, col in enumerate(df_gps.columns, 1):
    print(f"  {i:2d}. {col}")


✓ Columns reordered

Final column order (29 columns):
   1. id_obs
   2. bear_code
   3. bear_name
   4. sex
   5. age
   6. age_class
   7. year
   8. date_gmt
   9. time_gmt
  10. tracking_system
  11. system
  12. datum
  13. zone
  14. coordinate_system
  15. x
  16. y
  17. x_long
  18. y_lat
  19. country
  20. region
  21. activity
  22. altitude
  23. temp
  24. with_cubs_estimated
  25. n_cubs_estimated
  26. id_cub1
  27. id_cub2
  28. id_cub3
  29. remarks


In [111]:
# Summary of GPS data quality and values
print("=== GPS DATA SUMMARY ===\n")

# Data types
print("Data types:")
print(df_gps.dtypes)

# Missing values
print("\n\nMissing values:")
missing_data = df_gps.isnull().sum()
print(missing_data[missing_data > 0])

# Key variable summaries
print("\n\n=== KEY VARIABLE SUMMARIES ===\n")

print(f"ID_obs: {df_gps['id_obs'].nunique()} unique observations")
print(f"Bear_code: {df_gps['bear_code'].nunique()} unique bears (with {df_gps['bear_code'].isna().sum()} missing)")
print(f"Bear_name: {df_gps['bear_name'].nunique()} unique bear names")
print(f"  Top 5 bears: {df_gps['bear_name'].value_counts().head().to_dict()}")

print(f"\nSex distribution:")
print(df_gps['sex'].value_counts(dropna=False))

print(f"\nAge class distribution:")
print(df_gps['age_class'].value_counts(dropna=False))

print(f"\nYear range: {df_gps['year'].min()} - {df_gps['year'].max()}")
print(f"Year distribution:")
print(df_gps['year'].value_counts().sort_index())

print(f"\nCountry distribution:")
print(df_gps['country'].value_counts(dropna=False))

print(f"\nActivity distribution:")
print(df_gps['activity'].value_counts(dropna=False))

print(f"\nWith_cubs_estimated distribution:")
print(df_gps['with_cubs_estimated'].value_counts(dropna=False))

print(f"\nN_cubs_estimated distribution:")
print(df_gps['n_cubs_estimated'].value_counts(dropna=False).head(10))

print(f"\n\nCoordinate systems used:")
print(df_gps['coordinate_system'].value_counts(dropna=False))

print(f"\n\nBasic statistics for numeric columns:")
print(df_gps[['altitude', 'temp', 'age']].describe())


=== GPS DATA SUMMARY ===

Data types:
id_obs                          int64
bear_code                         str
bear_name                         str
sex                               str
age                             int64
age_class                         str
year                            int64
date_gmt               datetime64[us]
time_gmt                       object
tracking_system                   str
system                            str
datum                             str
zone                              str
coordinate_system                 str
x                               int64
y                               int64
x_long                        float64
y_lat                         float64
country                           str
region                            str
activity                      float64
altitude                      float64
temp                          float64
with_cubs_estimated            object
n_cubs_estimated                int64
id_cub1     

In [116]:
# Export GPS data to CSV
gps_output_file = OUTPUT_DIR / "GPS_female_bears_with_cubs.csv"

df_gps.to_csv(gps_output_file, index=False)

print(f"✓ GPS data exported to: {gps_output_file}")
print(f"  - Rows: {len(df_gps)}")
print(f"  - Columns: {len(df_gps.columns)}")
print(f"\nColumn list:")
for i, col in enumerate(df_gps.columns, 1):
    print(f"  {i:2d}. {col}")


✓ GPS data exported to: /home/aniol-garriga-torra-boss/Escriptori/ANIOL/UNI/4t Carrera/TFG/TFG-pirineus_raster/notebooks/cleaned_data/GPS_female_bears_with_cubs.csv
  - Rows: 4629
  - Columns: 29

Column list:
   1. id_obs
   2. bear_code
   3. bear_name
   4. sex
   5. age
   6. age_class
   7. year
   8. date_gmt
   9. time_gmt
  10. tracking_system
  11. system
  12. datum
  13. zone
  14. coordinate_system
  15. x
  16. y
  17. x_long
  18. y_lat
  19. country
  20. region
  21. activity
  22. altitude
  23. temp
  24. with_cubs_estimated
  25. n_cubs_estimated
  26. id_cub1
  27. id_cub2
  28. id_cub3
  29. remarks


In [113]:
# Preview of GPS data
print("First 5 rows of GPS female bears data:\n")
df_gps.head()


First 5 rows of GPS female bears data:



,id_obs,bear_code,bear_name,sex,age,age_class,year,date_gmt,time_gmt,tracking_system,...,region,activity,altitude,temp,with_cubs_estimated,n_cubs_estimated,id_cub1,id_cub2,id_cub3,remarks
0,36119,F018,Hvala,F,8,Adult,2007,2007-05-01,03:01:00,GPS,...,Haute-Garonne,NaN,NaN,NaN,<6month,2,Pollen,Bambou,-,NaN
1,36120,F018,Hvala,F,8,Adult,2007,2007-05-01,15:02:00,GPS,...,Haute-Garonne,NaN,NaN,NaN,<6month,2,Pollen,Bambou,-,NaN
2,36121,F018,Hvala,F,8,Adult,2007,2007-05-01,18:03:00,GPS,...,Haute-Garonne,NaN,NaN,NaN,<6month,2,Pollen,Bambou,-,NaN
3,36122,F018,Hvala,F,8,Adult,2007,2007-05-01,21:03:00,GPS,...,Haute-Garonne,NaN,NaN,NaN,<6month,2,Pollen,Bambou,-,NaN
4,36123,F018,Hvala,F,8,Adult,2007,2007-05-02,09:02:00,GPS,...,Haute-Garonne,NaN,NaN,NaN,<6month,2,Pollen,Bambou,-,NaN


## 7. Extract Observations Data from Third Sheet

In [117]:
# Load the third sheet (Observations female bears with cubs)
df_obs_raw = pd.read_excel(INPUT_FILE, sheet_name=2)

print("Observations Data - Third Sheet")
print(f"Shape: {df_obs_raw.shape}")
print(f"\nColumn names:")
for i, col in enumerate(df_obs_raw.columns, 1):
    print(f"  {i:2d}. {col}")
print(f"\nFirst few rows:")
df_obs_raw.head()


Observations Data - Third Sheet
Shape: (1962, 30)

Column names:
   1. id_obs
   2. confirmed_individual
   3. probable_individual
   4. sex
   5. age
   6. age_class
   7. year
   8. date
   9. date.precision
  10. x_long
  11. y_lat
  12. coordinates.precision
  13. site
  14. municipality
  15. region
  16. country
  17. with_cubs_observed
  18. n_cubs_observed
  19. with_cubs_estimated
  20. n_cubs_estimated
  21. id_cub1
  22. id_cub2
  23. id_cub3
  24. method
  25. obs_type
  26. hairtrap
  27. hour
  28. category
  29. database
  30. remarks

First few rows:


,id_obs,confirmed_individual,probable_individual,sex,age,age_class,year,date,date.precision,x_long,...,id_cub1,id_cub2,id_cub3,method,obs_type,hairtrap,hour,category,database,remarks
0,1270,Cannelle,Cannelle,F,15.0,Adult,2004,21/8/2004,most_accurate,-0.546776,...,M012,NaN,NaN,Oportunistic,Scat,NaN,NaN,1.0,recap_individualisation.xlxs,NaN
1,1300,Cannelle,Cannelle,F,15.0,Adult,2004,7/9/2004,most_accurate,-0.482955,...,M012,NaN,NaN,Systematic,Scat,NaN,NaN,1.0,recap_individualisation.xlxs,NaN
2,1305,Cannelle,Cannelle,F,15.0,Adult,2004,14/9/2004,most_accurate,-0.533117,...,M012,NaN,NaN,Oportunistic,Scat,NaN,NaN,1.0,recap_individualisation.xlxs,NaN
3,1319,Cannelle,Cannelle,F,15.0,Adult,2004,23/9/2004,register,-0.535927,...,M012,NaN,NaN,Sampling_station,Hair,1.0,NaN,1.0,Seguiment 2004,Hair sampling station
4,1320,Cannelle,Cannelle,F,15.0,Adult,2004,23/9/2004,register,-0.535927,...,M012,NaN,NaN,Sampling_station,Photo,NaN,NaN,1.0,Seguiment 2004,Hair sampling station


In [118]:
# Process observations data: convert column names to lowercase
df_obs = df_obs_raw.copy()

# Convert all column names to lowercase
df_obs.columns = df_obs.columns.str.lower()

print("✓ Column names converted to lowercase")
print(f"\nProcessed column names ({len(df_obs.columns)} columns):")
for i, col in enumerate(df_obs.columns, 1):
    print(f"  {i:2d}. {col}")


✓ Column names converted to lowercase

Processed column names (30 columns):
   1. id_obs
   2. confirmed_individual
   3. probable_individual
   4. sex
   5. age
   6. age_class
   7. year
   8. date
   9. date.precision
  10. x_long
  11. y_lat
  12. coordinates.precision
  13. site
  14. municipality
  15. region
  16. country
  17. with_cubs_observed
  18. n_cubs_observed
  19. with_cubs_estimated
  20. n_cubs_estimated
  21. id_cub1
  22. id_cub2
  23. id_cub3
  24. method
  25. obs_type
  26. hairtrap
  27. hour
  28. category
  29. database
  30. remarks


In [119]:
# Summary of observations data quality and values
print("=== OBSERVATIONS DATA SUMMARY ===\n")

# Data types
print("Data types:")
print(df_obs.dtypes)

# Missing values
print("\n\nMissing values:")
missing_data = df_obs.isnull().sum()
print(missing_data[missing_data > 0])

# Key variable summaries
print("\n\n=== KEY VARIABLE SUMMARIES ===\n")

print(f"ID_obs: {df_obs['id_obs'].nunique()} unique observations")
print(f"Total rows: {len(df_obs)}")

print(f"\nConfirmed individuals:")
print(f"  {df_obs['confirmed_individual'].notna().sum()} non-null values")
print(f"  {df_obs['confirmed_individual'].nunique()} unique values")

print(f"\nProbable individuals:")
print(f"  {df_obs['probable_individual'].notna().sum()} non-null values")
print(f"  {df_obs['probable_individual'].nunique()} unique values")

print(f"\nSex distribution:")
print(df_obs['sex'].value_counts(dropna=False))

print(f"\nAge class distribution:")
print(df_obs['age_class'].value_counts(dropna=False))

print(f"\nYear range: {df_obs['year'].min()} - {df_obs['year'].max()}")
print(f"Year distribution:")
print(df_obs['year'].value_counts().sort_index())

print(f"\nCountry distribution:")
print(df_obs['country'].value_counts(dropna=False))

print(f"\nRegion distribution:")
print(df_obs['region'].value_counts(dropna=False).head(10))

print(f"\nObservation type (obs_type) distribution:")
print(df_obs['obs_type'].value_counts(dropna=False))

print(f"\nMethod distribution:")
print(df_obs['method'].value_counts(dropna=False))

print(f"\nWith_cubs_observed distribution:")
print(df_obs['with_cubs_observed'].value_counts(dropna=False))

print(f"\nN_cubs_observed distribution:")
print(df_obs['n_cubs_observed'].value_counts(dropna=False).head(10))

print(f"\nWith_cubs_estimated distribution:")
print(df_obs['with_cubs_estimated'].value_counts(dropna=False))

print(f"\nN_cubs_estimated distribution:")
print(df_obs['n_cubs_estimated'].value_counts(dropna=False).head(10))

print(f"\nDatabase distribution:")
print(df_obs['database'].value_counts(dropna=False))

print(f"\nDate precision distribution:")
print(df_obs['date.precision'].value_counts(dropna=False))

print(f"\nCoordinates precision distribution:")
print(df_obs['coordinates.precision'].value_counts(dropna=False))

print(f"\nCategory distribution:")
print(df_obs['category'].value_counts(dropna=False).head(10))

print(f"\n\nBasic statistics for numeric columns:")
print(df_obs[['age', 'hour']].describe())


=== OBSERVATIONS DATA SUMMARY ===

Data types:
id_obs                     int64
confirmed_individual         str
probable_individual          str
sex                          str
age                      float64
age_class                    str
year                       int64
date                         str
date.precision               str
x_long                   float64
y_lat                    float64
coordinates.precision    float64
site                         str
municipality                 str
region                       str
country                      str
with_cubs_observed       float64
n_cubs_observed          float64
with_cubs_estimated          str
n_cubs_estimated         float64
id_cub1                      str
id_cub2                      str
id_cub3                      str
method                       str
obs_type                     str
hairtrap                 float64
hour                         str
category                 float64
database                     

In [120]:
# Export observations data to CSV
obs_output_file = OUTPUT_DIR / "observations_female_bears_with_cubs.csv"

df_obs.to_csv(obs_output_file, index=False)

print(f"✓ Observations data exported to: {obs_output_file}")
print(f"  - Rows: {len(df_obs)}")
print(f"  - Columns: {len(df_obs.columns)}")
print(f"\nColumn list:")
for i, col in enumerate(df_obs.columns, 1):
    print(f"  {i:2d}. {col}")


✓ Observations data exported to: /home/aniol-garriga-torra-boss/Escriptori/ANIOL/UNI/4t Carrera/TFG/TFG-pirineus_raster/notebooks/cleaned_data/observations_female_bears_with_cubs.csv
  - Rows: 1962
  - Columns: 30

Column list:
   1. id_obs
   2. confirmed_individual
   3. probable_individual
   4. sex
   5. age
   6. age_class
   7. year
   8. date
   9. date.precision
  10. x_long
  11. y_lat
  12. coordinates.precision
  13. site
  14. municipality
  15. region
  16. country
  17. with_cubs_observed
  18. n_cubs_observed
  19. with_cubs_estimated
  20. n_cubs_estimated
  21. id_cub1
  22. id_cub2
  23. id_cub3
  24. method
  25. obs_type
  26. hairtrap
  27. hour
  28. category
  29. database
  30. remarks


In [121]:
# Preview of observations data
print("First 5 rows of observations female bears data:\n")
df_obs.head()


First 5 rows of observations female bears data:



,id_obs,confirmed_individual,probable_individual,sex,age,age_class,year,date,date.precision,x_long,...,id_cub1,id_cub2,id_cub3,method,obs_type,hairtrap,hour,category,database,remarks
0,1270,Cannelle,Cannelle,F,15.0,Adult,2004,21/8/2004,most_accurate,-0.546776,...,M012,NaN,NaN,Oportunistic,Scat,NaN,NaN,1.0,recap_individualisation.xlxs,NaN
1,1300,Cannelle,Cannelle,F,15.0,Adult,2004,7/9/2004,most_accurate,-0.482955,...,M012,NaN,NaN,Systematic,Scat,NaN,NaN,1.0,recap_individualisation.xlxs,NaN
2,1305,Cannelle,Cannelle,F,15.0,Adult,2004,14/9/2004,most_accurate,-0.533117,...,M012,NaN,NaN,Oportunistic,Scat,NaN,NaN,1.0,recap_individualisation.xlxs,NaN
3,1319,Cannelle,Cannelle,F,15.0,Adult,2004,23/9/2004,register,-0.535927,...,M012,NaN,NaN,Sampling_station,Hair,1.0,NaN,1.0,Seguiment 2004,Hair sampling station
4,1320,Cannelle,Cannelle,F,15.0,Adult,2004,23/9/2004,register,-0.535927,...,M012,NaN,NaN,Sampling_station,Photo,NaN,NaN,1.0,Seguiment 2004,Hair sampling station


In [114]:
# Display first rows of cleaned data
print("First 10 rows of cleaned bear individual data:\n")
df_clean.head(10)

First 10 rows of cleaned bear individual data:



,code,name,genotype,genotype_uab,sex,born_year,mum_name,mum_code,father_name,father_code,...,num_detections_may_24,num_detections_jun_24,num_detections_jul_24,num_detections_aug_24,num_detections_sep_24,num_detections_oct_24,num_detections_nov_24,num_detections_dec_24,num_detections_total_24,disappeared
0,Papillon,Papillon,NaN,NaN,M,NaN,NaN,NaN,NaN,NaN,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0,1
1,Cannelle,Cannelle,S2-PYR6,NaN,F,NaN,NaN,NaN,NaN,NaN,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0,1
2,Camille,Camille/Aspe-Ouest,S1-PYR4,Camille,M,1998,Cannelle,Sensecodi,Papillon,Sensecodi,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0,1
3,Ourson,Oursonmort,NaN,NaN,M,2000,Cannelle,Sensecodi,NaN,NaN,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0,1
4,F001,Ziva,S8-SLO13,NaN,F,1990,Slovène,Sensecodi,Slovène,Sensecodi,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0,1
5,F002,Melba,NaN,NaN,F,1991,Slovène,Sensecodi,Slovène,Sensecodi,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0,1
6,M003,Pyros,S1-SLO1,NaN,M,1988,Slovène,Sensecodi,Slovène,Sensecodi,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0,1
7,M004,Nere,S2-SL06,NaN,M,1997,Ziva,F001,Slovène,Sensecodi,...,3,5,4,1,1,1,<NA>,<NA>,23,0
8,M005,Kouki,NaN,NaN,M,1997,Ziva,F001,Pyros,M003,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0,1
9,M006,Boutxy,S1-SLO2,NaN,M,1997,Mellba,F002,Pyros,M003,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0,1
